# NMF Topic Model — England k=30 No Media

**Corpus:** 3,931 articles post-cleaning (3,943 raw → cleaned by removing GOV.UK footer phrases, FFT newsletter boilerplate, and articles <200 chars)

**Sources:** SchoolsWeek (~69%), GOV.UK, FFT, EPI, Nuffield, FED

**Model:** NMF, k=30, max_features=3000, min_df=3, init=nndsvd, max_iter=1000

**Purpose:** One of four England model variants trained for the contestability demonstration. Same k=30 and preprocessing as the production model (`nmf_eng_30`), but with SchoolsWeek removed from the corpus (~69% of articles). This reveals what the English education policy debate looks like without the dominant media voice — which topics disappear, which emerge, and how much of the "debate" was actually one outlet's editorial agenda. See `docs/internal/current/technical/contestability_example.md`.

**Key finding:** TODO — run the notebook and document what changes when SchoolsWeek is removed. Which topics disappear (media-driven)? Which emerge or strengthen (previously drowned out)? Which stay stable (robust across source composition)?


# 0. Imports 

In [1]:
import sys
sys.path.insert(0, "/workspaces/AM1_topic_modelling")

import logging
import pandas as pd
import numpy as np
from pathlib import Path

logging.basicConfig(level=logging.INFO)

from model_pipeline.training.s02_cleaning import run_cleaning
from model_pipeline.training.s03_spacy_processing import run_spacy_processing
from model_pipeline.training.s04_vectorisation import run_vectorisation, build_vectorizer
from model_pipeline.training.s05_nmf_training import run_nmf_training, get_top_words_per_topic
from model_pipeline.training.s06_topic_allocation import TOPIC_NAMES

import logging
logging.getLogger("gensim").setLevel(logging.WARNING)

from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

INFO:model_pipeline.training.s06_topic_allocation:Loaded 5 topic names from llm_topic_review.json


# 1. Load England training data

In [2]:
preprocessed_path = Path("/workspaces/AM1_topic_modelling/data/training/eng_training_nm_preprocessed.parquet")

if preprocessed_path.exists():
    print("Loading preprocessed data (skipping cleaning + spaCy)...")
    df = pd.read_parquet(preprocessed_path)
    SKIP_PREPROCESSING = True
else:
    csv_path = Path("/workspaces/AM1_topic_modelling/data/training/eng_training_nm.csv")
    df = pd.read_csv(csv_path)
    SKIP_PREPROCESSING = False

print(f"Loaded: {df.shape}")
print(f"Skip preprocessing: {SKIP_PREPROCESSING}")

Loading preprocessed data (skipping cleaning + spaCy)...
Loaded: (1198, 18)
Skip preprocessing: True


# 2. Prepare text column

The Supabase schema has `title` and `text` as separate columns. The pipeline expects a combined `text` column. Also rename `article_date` → `date` to match pipeline expectations.

In [3]:
if not SKIP_PREPROCESSING:
    # Combine title + text (same as s01_data_loader)
    df["text"] = df["title"].fillna("") + "\n\n" + df["text"].fillna("")
    df["date"] = pd.to_datetime(df["article_date"], errors="coerce")
    print(f"Text combined. Sample length: {df['text'].str.len().describe()}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


## 3. Cleaning (s02)

In [4]:
if not SKIP_PREPROCESSING:
    df = run_cleaning(df)
    print(f"After cleaning: {df.shape}")
    print(f"Empty text_clean: {(df['text_clean'].str.len() == 0).sum()}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


# 4. spaCy processing (s03)

This takes a few minutes on ~4k articles.

In [5]:
if not SKIP_PREPROCESSING:
    df = run_spacy_processing(df)
    print(f"After spaCy: {df.shape}")
    print(f"Empty text_final: {(df['text_final'].str.len() == 0).sum()}")
    print(f"Avg tokens per doc: {df['tokens_final'].apply(len).mean():.0f}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


In [6]:
if not SKIP_PREPROCESSING:
    df.to_parquet("/workspaces/AM1_topic_modelling/data/training/eng_training_nm_preprocessed.parquet")
    print(f"Saved preprocessed data: {df.shape}")
else:
    print("Already loaded from parquet")

Already loaded from parquet


# 5. TF-IDF vectorisation (s04)

In [7]:
# Using same params as config.yaml: min_df=3, max_df=0.85, max_features=3000, ngram_range=(1,2)
vec_out = run_vectorisation(df)
print(f"TF-IDF matrix: {vec_out.X.shape}")
print(f"Vocabulary size: {len(vec_out.feature_names)}")
print(f"Sample features: {vec_out.feature_names[:20].tolist()}")

INFO:model_pipeline.training.s04_vectorisation:Step 04 (vectorisation): starting. Input shape=(1198, 18)
INFO:model_pipeline.training.s04_vectorisation:TF-IDF shape: (1198, 3000)
INFO:model_pipeline.training.s04_vectorisation:Vectorizer params: min_df=3 max_df=0.85 max_features=3000 ngram_range=(1, 2)
INFO:model_pipeline.training.s04_vectorisation:Sample features: ['ab', 'ability', 'able', 'absence', 'absence absence', 'absence attainment', 'absence code', 'absence disadvantaged', 'absence exclusion', 'absence illness', 'absence london', 'absence persistent', 'absence pre', 'absence primary', 'absence pupil', 'absence school', 'absence secondary', 'absence session', 'absent', 'absent pupil']


TF-IDF matrix: (1198, 3000)
Vocabulary size: 3000
Sample features: ['ab', 'ability', 'able', 'absence', 'absence absence', 'absence attainment', 'absence code', 'absence disadvantaged', 'absence exclusion', 'absence illness', 'absence london', 'absence persistent', 'absence pre', 'absence primary', 'absence pupil', 'absence school', 'absence secondary', 'absence session', 'absent', 'absent pupil']


# 6. Train NMF (k=30)

In [8]:
nmf_out = run_nmf_training(vec_out.X, n_topics=30, random_state=42, init="nndsvd", max_iter=1000)
print(f"W shape: {nmf_out.W.shape}")
print(f"H shape: {nmf_out.H.shape}")
print(f"Reconstruction error: {nmf_out.reconstruction_error:.6f}")


INFO:model_pipeline.training.s05_nmf_training:Step 05 (NMF): starting. X shape=(1198, 3000)
INFO:model_pipeline.training.s05_nmf_training:Step 05 (NMF): complete.
INFO:model_pipeline.training.s05_nmf_training:W shape=(1198, 30) | H shape=(30, 3000)
INFO:model_pipeline.training.s05_nmf_training:Reconstruction error: 26.379661


W shape: (1198, 30)
H shape: (30, 3000)
Reconstruction error: 26.379661


# 7. Topic stability across random seeds

In [9]:
seeds = [42, 123, 456, 789, 1024]
H_matrices = []

for seed in seeds:
    model = NMF(n_components=30, init="nndsvda", random_state=seed, max_iter=1000)
    model.fit(vec_out.X)
    H_matrices.append(model.components_)
    print(f"Seed {seed}: recon error = {model.reconstruction_err_:.4f}")

# Compare all pairs of H matrices
pair_scores = []
for i in range(len(seeds)):
    for j in range(i + 1, len(seeds)):
        sim = cosine_similarity(H_matrices[i], H_matrices[j])
        # Best match for each topic in run i against run j
        best_matches = sim.max(axis=1).mean()
        pair_scores.append(best_matches)
        print(f"Seeds {seeds[i]} vs {seeds[j]}: avg best-match similarity = {best_matches:.4f}")

avg_stability = np.mean(pair_scores)
print(f"\nOverall topic stability: {avg_stability:.4f}")
print(f"Previous stability (old model): 0.94")
print(f"Interpretation: >0.90 = highly stable, 0.80-0.90 = acceptable, <0.80 = unstable")


Seed 42: recon error = 26.3797
Seed 123: recon error = 26.3655
Seed 456: recon error = 26.3522
Seed 789: recon error = 26.3585
Seed 1024: recon error = 26.3489
Seeds 42 vs 123: avg best-match similarity = 0.9636
Seeds 42 vs 456: avg best-match similarity = 0.9791
Seeds 42 vs 789: avg best-match similarity = 0.9315
Seeds 42 vs 1024: avg best-match similarity = 0.9554
Seeds 123 vs 456: avg best-match similarity = 0.9732
Seeds 123 vs 789: avg best-match similarity = 0.9505
Seeds 123 vs 1024: avg best-match similarity = 0.9517
Seeds 456 vs 789: avg best-match similarity = 0.9517
Seeds 456 vs 1024: avg best-match similarity = 0.9756
Seeds 789 vs 1024: avg best-match similarity = 0.9758

Overall topic stability: 0.9608
Previous stability (old model): 0.94
Interpretation: >0.90 = highly stable, 0.80-0.90 = acceptable, <0.80 = unstable


#### TODO: Document stability findings for k=30 no-media model.


# 8. Inspect topics — top words per topic

Compare with previous topic names to see if they still hold.

In [10]:
topics = get_top_words_per_topic(nmf_out.nmf_model, vec_out.feature_names, n_top_words=100)

# Display stopwords — these don't affect the model, just make the top words easier to read
display_stop = {
    "school", "education", "pupil", "student", "teacher", "year", "new", "work",
    "time", "say", "make", "good", "need", "use", "know", "want", "come", "take",
    "people", "government", "report", "system", "support", "include", "provide",
    "number", "change", "part", "set", "high", "low", "level", "national", "local",
    "public", "service", "also", "would", "could", "one", "two", "first", "last",
    "week", "month", "day", "told", "said", "according", "cent", "per", "per cent",
    "child", "children", "young", "staff", "area", "programme", "policy",
    "guidance", "framework", "response", "statement", "proposal", "approach",
    "review", "update", "document", "detail", "section", "datum", "figure",
    "survey", "rate", "score", "point", "proportion", "percentage",
    "organisation", "department", "committee", "institute", "foundation",
    "summit", "voice", "stakeholder", "partnership", "engagement",
    "scheme", "initiative", "pilot", "introduce", "implement", "launch",
    "office", "official", "notification", "recipient", "correspondence",
    "cookie", "banner", "subscribe", "contact", "submit", "accessibility",
    "share", "print", "visit", "site", "experience", "article", "news", "blog",
    "interesting", "fact", "previous", "current", "date", "information",
    "different", "large", "place", "individual", "view", "analysis",
    "thing", "way", "job"
}

In [11]:
topics = get_top_words_per_topic(nmf_out.nmf_model, vec_out.feature_names, n_top_words=30)

for topic_id, words in enumerate(topics):
    print(f"\nTopic {topic_id}: {', '.join(words)}")


Topic 0: late, late information, local authority, authority, academy, academy school, education, local, college local, esfa late, agency academy, html details, details education, details, skill agency, esfa, information, education late, education skill, information action, academy late, action education, authority education, agency, school college, provider html, html, education provider, late academy, late local

Topic 1: child, family, need, care, support, early, life, child school, standard, send, right, system, service, child child, attendance, government, poverty, vulnerable, school child, sure, inclusion, child young, social, school, child social, special, reading, area, classroom, social care

Topic 2: notice, warning notice, warning, regional director, dfe regional, director, regional, dfe, reason issue, urn notice, notice warning, urn, academy, judgement dfe, trust reason, relation, trust relation, trust, judgement, reason, director dfe, council, issue, issue inadequate, inad

In [12]:
# # TODO: assign names after inspecting top words
# TOPIC_NAMES_30_NM = {i: f"topic_{i}" for i in range(30)}

# df["dominant_topic"] = nmf_out.W.argmax(axis=1)
# df["dominant_topic_name"] = df["dominant_topic"].map(TOPIC_NAMES_30_NM)
# df["dominant_topic_name"].value_counts()

# 9. Explore a topic — top articles + source concentration

In [13]:
# def explore_topic(topic_id, n=5):
#     """Show top N articles for a given topic, ranked by topic weight."""
#     W = nmf_out.W
#     topic_weights = W[:, topic_id]
#     top_idx = topic_weights.argsort()[::-1][:n]

#     # Topic words
#     words = topics[topic_id]
#     filtered = [w for w in words if w not in display_stop and len(w) > 2][:20]
#     print(f"TOPIC {topic_id} ({TOPIC_NAMES_30_NM[topic_id]}) — top words: {', '.join(filtered)}")
#     print(f"{'='*120}\n")

#     for rank, idx in enumerate(top_idx, 1):
#         row = df.iloc[idx]
#         weight = topic_weights[idx]
#         title = row.get("title", "No title")
#         source = row.get("source", "Unknown")
#         date = str(row.get("article_date", ""))[:10]
#         text = row.get("text_clean", row.get("text", ""))
#         if isinstance(text, str) and len(text) > 500:
#             text = text[:500] + "..."

#         print(f"[{rank}] weight={weight:.4f} | {source} | {date}")
#         print(f"    {title}")
#         print(f"    {text}\n")

# # Source concentration across all topics
# print("Source concentration (top 50 articles per topic):")
# print("=" * 80)
# for t in range(nmf_out.nmf_model.n_components):
#     top_idx = nmf_out.W[:, t].argsort()[::-1][:50]
#     breakdown = df.iloc[top_idx]['source'].value_counts()
#     pct = (breakdown / breakdown.sum() * 100).round(0).astype(int)
#     summary = ", ".join(f"{src} {p}%" for src, p in pct.items())
#     print(f"Topic {t:>2} ({TOPIC_NAMES_30_NM[t]}): {summary}")

# # Usage: explore_topic(0) for topic 0, explore_topic(3, n=10) for 10 articles


#### TODO: Document source concentration findings for the no-media model. Which topics changed? Which disappeared? Which emerged?

#### TODO: Document contestability implications of removing the dominant media source.


# 10. LLM-suggested topic names

Send top words per topic to Claude for naming suggestions. Compare with existing names.

In [14]:
# import os
# from pathlib import Path
# from dotenv import load_dotenv
# from anthropic import Anthropic
# import json
# import re

# load_dotenv(Path("/workspaces/AM1_topic_modelling/.env"))
# client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# # Build keyword lists (without display stopwords)
# topic_keyword_lines = []
# for i, words in enumerate(topics):
#     filtered = [w for w in words if w not in display_stop and len(w) > 2][:30]
#     topic_keyword_lines.append(f"Topic {i}: {', '.join(filtered)}")

# my_name_lines = []
# for i in range(30):
#     my_name_lines.append(f"Topic {i}: proposed name is '{TOPIC_NAMES_30_NM.get(i, 'unknown')}'")

# prompt = f"""You are helping label topics from an NMF topic model trained on UK education policy documents (2023-2025).
# The corpus includes articles from government departments, think tanks, media outlets, and research organisations in England.
# This model has k=30 topics. The corpus has SchoolsWeek removed (no media variant).

# STEP 1: For each topic below, suggest a name based ONLY on the keywords. Do not look ahead to Step 2.

# {chr(10).join(topic_keyword_lines)}

# For each topic return:
# - suggested_name (2-4 words, snake_case)
# - explanation (one sentence)

# STEP 2: Now compare your suggestions against these human-proposed names for the same topics:

# {chr(10).join(my_name_lines)}

# For each topic, assess:
# - proposed_name_assessment: "AGREE" (proposed name fits the keywords well), "CLOSE" (reasonable but could be more precise), or "DISAGREE" (proposed name doesn't capture the topic well)
# - proposed_name_note: one sentence explaining why

# Return ONLY a JSON list combining both steps:
# [
#   {{"topic": 0, "suggested_name": "name", "explanation": "why", "proposed_name_assessment": "AGREE", "proposed_name_note": "why"}},
#   ...
# ]
# No other text, no markdown, no code fences."""

# response = client.messages.create(
#     model="claude-sonnet-4-20250514",
#     max_tokens=4096,
#     messages=[{"role": "user", "content": prompt}],
# )

# llm_text = response.content[0].text

# # Strip markdown code fences if present
# cleaned = re.sub(r'^```(?:json)?\n?', '', llm_text.strip())
# cleaned = re.sub(r'\n?```$', '', cleaned.strip())
# llm_results = json.loads(cleaned)

# # Summary table
# print(f"{'Topic':>5}  {'My Name':<35}  {'Status':<8}  {'LLM Suggestion':<35}  Notes")
# print("=" * 140)
# for r in llm_results:
#     i = r["topic"]
#     mine = TOPIC_NAMES_30_NM.get(i, "???")
#     print(f"{i:>5}  {mine:<35}  {r['proposed_name_assessment']:<8}  {r['suggested_name']:<35}  {r['proposed_name_note']}")

# # Detailed explanations
# print("\n\nDETAILED EXPLANATIONS:")
# print("=" * 80)
# for r in llm_results:
#     print(f"\nTopic {r['topic']}: {r['suggested_name']}")
#     print(f"  Why: {r['explanation']}")
#     print(f"  My name '{TOPIC_NAMES_30_NM.get(r['topic'], '???')}': {r['proposed_name_assessment']} — {r['proposed_name_note']}")

# # Save
# with open("/workspaces/AM1_topic_modelling/data/evaluation_outputs/llm_topic_review_k30_nm.json", "w") as f:
#     json.dump(llm_results, f, indent=2)
# print("\nSaved to data/evaluation_outputs/llm_topic_review_k30_nm.json")


# 11. Topic distribution — how many articles per dominant topic?

In [17]:
dominant_topics = nmf_out.W.argmax(axis=1)
dominant_weights = nmf_out.W.max(axis=1)

topic_counts = pd.Series(dominant_topics).value_counts().sort_index()
topic_df = pd.DataFrame({
    "topic_num": topic_counts.index,
    "topic_name": [TOPIC_NAMES.get(i, "???") for i in topic_counts.index],
    "count": topic_counts.values,
    "pct": (topic_counts.values / len(dominant_topics) * 100).round(1),
})
print(topic_df.to_string(index=False))
print(f"\nDominant weight — min: {dominant_weights.min():.4f}, mean: {dominant_weights.mean():.4f}, max: {dominant_weights.max():.4f}")


 topic_num                     topic_name  count  pct
         0 education_attendance_provision     65  5.4
         1   academy_oversight_management     50  4.2
         2         academy_trust_warnings     67  5.6
         3            teacher_pay_strikes     84  7.0
         4     ofsted_inspection_concerns     47  3.9
         5                            ???     32  2.7
         6                            ???     34  2.8
         7                            ???     54  4.5
         8                            ???     57  4.8
         9                            ???     75  6.3
        10                            ???     45  3.8
        11                            ???     25  2.1
        12                            ???     44  3.7
        13                            ???     21  1.8
        14                            ???     76  6.3
        15                            ???     14  1.2
        16                            ???     43  3.6
        17                  

#### TODO: Document topic distribution findings. Compare to nmf_eng_30 (full corpus).

# 12. Topic distribution by source

Check if any source dominates a topic (specification choice: corpus composition).

In [18]:
df["dominant_topic"] = dominant_topics
df["dominant_topic_name"] = df["dominant_topic"].map(TOPIC_NAMES_30_NM)
ct = pd.crosstab(df["source"], df["dominant_topic_name"], normalize="columns").round(2)
print("Source distribution per topic (column-normalised):")
ct


NameError: name 'TOPIC_NAMES_30_NM' is not defined

#### TODO: Document source-topic heatmap findings. Without SchoolsWeek, how does source distribution change? Do smaller sources (EPI, Nuffield, FFT) become visible?


# 13. Save retrained model artifacts

In [ ]:
import json
from datetime import datetime

run_id = datetime.now().strftime("%Y-%m-%d_%H%M%S")
run_dir = Path(f"/workspaces/AM1_topic_modelling/experiments/outputs/runs/{run_id}")
run_dir.mkdir(parents=True, exist_ok=True)

# Topic summary for sensitivity page
topic_summary = {
    "model_id": "nmf_eng_30_nm",
    "n_topics": 5,
    "n_articles": len(df),
    "topics": [
        {"topic_num": i, "name": TOPIC_NAMES_30_NM[i], "count": int((nmf_out.W.argmax(axis=1) == i).sum())}
        for i in range(5)
    ]
}
with open(run_dir / "topic_summary.json", "w") as f:
    json.dump(topic_summary, f, indent=2)

# Topic names
with open(run_dir / "topic_names.json", "w") as f:
    json.dump(TOPIC_NAMES_30_NM, f, indent=2)

# Run metadata
metadata = {
    "run_id": run_id,
    "model_id": "nmf_eng_30_nm",
    "n_articles": len(df),
    "n_topics": 5,
    "init": "nndsvd",
    "random_state": 42,
    "max_iter": 1000,
    "reconstruction_error": float(nmf_out.reconstruction_error),
    "corpus": "eng_training_3943_supabase",
    "note": "England k=5 for contestability comparison. Not a production model."
}
with open(run_dir / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to {run_dir}")


# 15. Save dashboard summary JSON (for Specification Sensitivity page)

In [ ]:
import json

dominant = nmf_out.W.argmax(axis=1)

topic_data = []
for i in range(30):
    mask = dominant == i
    count = int(mask.sum())
    source_counts = df.loc[mask, "source"].value_counts()
    top_source = source_counts.index[0] if len(source_counts) > 0 else "unknown"
    top_source_pct = round(float(source_counts.iloc[0] / source_counts.sum()), 2) if len(source_counts) > 0 else 0

    topic_data.append({
        "topic_num": i,
        "name": TOPIC_NAMES_30_NM[i],
        "count": count,
        "pct": round(count / len(df) * 100, 1),
        "top_source": top_source,
        "top_source_pct": top_source_pct,
        "single_source": top_source_pct > 0.90,
    })

summary = {
    "model_id": "nmf_eng_30_nm",
    "n_topics": 30,
    "n_articles": len(df),
    "metrics": {
        "reconstruction_error": round(float(nmf_out.reconstruction_error), 4),
        "stability": round(float(avg_stability), 4),
        "mean_dominant_weight": round(float(nmf_out.W.max(axis=1).mean()), 4),
        "max_dominant_weight": round(float(nmf_out.W.max(axis=1).max()), 4),
    },
    "topics": topic_data,
}

out_path = "/workspaces/AM1_topic_modelling/data/evaluation_outputs/nmf_eng_30_nm_summary.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved to {out_path}")
print(f"Single-source topics: {sum(1 for t in topic_data if t['single_source'])}/30")


## Final Summary: England k=30 No Media — Source Composition Sensitivity

TODO: Complete after running the notebook.

This notebook trained an NMF topic model on the England corpus with SchoolsWeek removed (~1,200 articles from ~3,931). Same k=30, same preprocessing, same model architecture as the production model — only the corpus composition changed.

**Key questions this answers:**
- Which topics disappear entirely when media is removed (media-driven topics)?
- Which topics emerge or strengthen without media (previously drowned out by SchoolsWeek volume)?
- Which topics stay stable (robust across source composition)?
- How much of the "education debate" was actually one outlet's editorial agenda?

This model is not intended for production use. Its purpose is to provide pre-computed results for the Specification Sensitivity dashboard page, where toggling "Include media sources" on/off shows what changes. See `docs/internal/current/technical/contestability_example.md` for the full contestability analysis.
